# SCVI integrate subsets

## Python modules

In [1]:
import scanpy as sc
import anndata as ad

import scanorama
import scvi

from scipy import sparse
import pandas as pd
import numpy as np

import pickle

import seaborn as sb
import matplotlib.pyplot as plt

import os

[rank: 0] Global seed set to 0
/nobackup/peer/fdeckert/miniconda3/envs/p.3.8.12-FD20220623HSCT/lib/python3.8/site-packages/flax/struct.py:133: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
/nobackup/peer/fdeckert/miniconda3/envs/p.3.8.12-FD20220623HSCT/lib/python3.8/site-packages/flax/struct.py:133: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
/nobackup/peer/fdeckert/miniconda3/envs/p.3.8.12-FD20220623HSCT/lib/python3.8/site-packages/flax/struct.py:133: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)


## Settup rpy2 

In [2]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [3]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

In [4]:
%load_ext rpy2.ipython

## Settings

In [5]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [6]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')

In [7]:
# Warnings R
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR) 

In [8]:
scvi.settings.num_threads=32

## Plotting 

In [9]:
# Scanpy figures
sc.set_figure_params(figsize=(7, 7), dpi_save=1200, fontsize=12, frameon=False, color_map='viridis', facecolor="white")

In [10]:
# Matplot figures
plt.rcParams['figure.figsize']=(7, 7)

In [11]:
# Load color 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}

In [12]:
# Set color function 
def set_color(categories, adata): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]
    
    for category in categories: 
        
        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)
        
    return adata

# Import AnnData 

In [13]:
adata = sc.read_h5ad('data/object/pp.h5ad')

In [62]:
adata_1 = sc.read_h5ad('data/object/subsets/annotation/lymphocyte.h5ad')
adata_2 = sc.read_h5ad('data/object/subsets/annotation/myeloid.h5ad')
adata_3 = sc.read_h5ad('data/object/subsets/annotation/structural.h5ad')

In [63]:
cell_type_leiden_subset = pd.concat([adata_1.obs, adata_2.obs, adata_3.obs])[['leiden_cell_type_main']].rename(columns={'leiden_cell_type_main':'cell_type_leiden_subset'})
adata.obs = adata.obs.merge(cell_type_leiden_subset, left_index=True, right_index=True, how='left')

In [64]:
adata = adata[adata.obs_names.isin(cell_type_leiden_subset.index)]
adata = adata[:, (adata.X>=3).sum(axis=0)>=3]

In [65]:
adata = adata.copy()

# SCVI integration 

In [ ]:
scvi.model.SCVI.setup_anndata(
    
    adata,
    batch_key='sample_name', 
    continuous_covariate_keys=['msS_RNA', 'msG2M_RNA', 'pMT_RNA']

)

In [ ]:
model = scvi.model.SCVI(
    
    adata, 
    n_latent=10, 
    n_hidden=128, 
    n_layers=2, 
    gene_likelihood='nb'
    
)

In [ ]:
model.train()

In [ ]:
model.save('data/object/subsets/scvi/model/', overwrite=True)

In [ ]:
adata.obsm['X_scvi'] = model.get_latent_representation()

# Dim reduction 

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, use_rep='X_scvi')
sc.tl.umap(adata, min_dist=1)

In [ ]:
adata = set_color(list(color.keys()), adata)

In [ ]:
sc.pl.umap(adata, color=['cell_type_leiden', 'cell_type_leiden_subset', 'patient_id', 'hto_demux_class', 'time_point', 'cc_phase_class', 'digestion_protocol', 'genotype_class', 'cc_aa_alignment_size'], size=20, ncols=4, wspace=0.4)

# Save results  

In [ ]:
adata.write_h5ad('data/object/subsets/scvi/adata.h5ad')

In [ ]:
pd.DataFrame(adata.obsm['X_umap'], index=adata.obs.index).to_csv('data/object/subsets/scvi/components/umap.csv')